# REDGARDEN Arena Bot AI — Unsupervised Pretraining

Runs on Google Colab (T4 GPU). Unsupervised pretrains GPT-2 small (124M) on the REDGARDEN
arena AI training corpus — `NORTHSTAR.md` §18.4's own next buildable step (S170-194,
founder: "do the work to prepare for unsupervised learning" / "target torch training on
colab").

**Open this directly from GitHub — no upload needed:** Colab → File → Open notebook →
GitHub tab → `emilyspringerton/REDGARDEN` →
`notebooks/redgarden_gpt2_pretrain_colab.ipynb`.

**This is genuinely unsupervised, not the later supervised fine-tune.** Next-token
prediction over raw `self`/`foe` arena state + action text (`packages/simulation/
arena_ai_bridge.c`'s own `arena_corpus_record()` output, one record per active hero per
tick) — no win/loss label, no reward signal, learning "what tends to happen next" across
every hero's replay data undifferentiated. The resulting checkpoint is the STARTING
WEIGHTS for `NORTHSTAR.md` §12 Phase E's own already-planned supervised, NORN-graded
fine-tune (Milestone 7+), not a finished game-playing policy by itself.

**Before running this:** build the corpus locally and sync it to Drive:
```bash
# after playing/running some real matches, so var/corpus/ has real data
python3 scripts/build_ai_corpus.py --min-records 1000
# then copy var/corpus/combined.jsonl to Drive as redgarden-corpus.jsonl
# under the DRIVE_FOLDER path the bootstrap cell below sets
```

**Paste-once workflow**, same pattern the sibling `gpt2-alpine-c` repo's own notebook
already uses: the one code cell below is all you ever paste into Colab. Hit play, approve
the Drive OAuth prompt when it appears, and it handles everything — clones (or pulls)
REDGARDEN, then runs `scripts/colab_train.py` for the actual training. All training logic
lives in that script, in git, not in this notebook — when the training approach changes,
it ships as a commit, and the *same* bootstrap cell picks it up on the next run via
`git pull`. Nothing to re-paste, no cells to manually resync.

In [ ]:
# === REDGARDEN arena bot AI unsupervised pretrain — reusable bootstrap cell ===
# This cell is the only thing you ever need to paste into Colab. It mounts
# Drive (approve the OAuth prompt when it appears), then pulls the latest
# training logic from git and runs it. Future changes to how training works
# ship as commits to scripts/colab_train.py — re-running this same cell
# always executes the current version, no re-pasting required.

from google.colab import drive
drive.mount('/content/drive')

import os
import subprocess

REPO_URL = 'https://github.com/emilyspringerton/REDGARDEN.git'
REPO_DIR = '/content/REDGARDEN'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

# Adjust DRIVE_FOLDER here only if your Drive layout differs from the default.
os.environ.setdefault('DRIVE_FOLDER', '/content/drive/MyDrive/redgarden-training')

subprocess.run(
    ['python3', 'scripts/colab_train.py'],
    cwd=REPO_DIR, check=True,
)

## Next Steps

After training:
1. Download `checkpoint-unsupervised-pretrain.tar.gz` from Drive.
2. Keep it as the starting point for §12 Phase E's own later supervised, NORN-graded
   fine-tune stage (that stage starts the model from this checkpoint, not the base `gpt2`
   checkpoint cold) — not wired up yet, a separate future pass.
3. File a completion Apple:
   ```bash
   emily apples post -t completion -repo REDGARDEN "Arena AI unsupervised pretrain complete" "..."
   ```
4. Update the relevant `EMILY/BACKLOG.md` item.